# MagneticTurbulence — restart persistence of magnetic unknowns

Verifies that `k_b`, `epsilon_b` and the magnetic field survive a save/restart
with identical continuation: a pre-run solves 0..t_half and saves a `.sauv`,
the restart case resumes to t_final, and the result is compared against a
continuous run 0..t_final.


In [ ]:
from pathlib import Path
import numpy as np
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()
run.initBuildDirectory()
BUILD = run.BUILD_DIRECTORY

NX, NY = 33, 33
DT = 0.0625
T_HALF = 0.25
T_FINAL = 0.5

def pre_run(d, n):
    name = n.replace("_restart", "_pre") + ".data"
    case = run.TRUSTCase(".", "magnetic_turbulence_restart_pre.data").copy(
        name, targetDirectory=d + "/pre_run")
    case.substitute_template(dict(nx=NX, ny=NY, dt=DT, t_half=T_HALF))
    case.run()

restart_case = run.addCaseFromTemplate(
    "magnetic_turbulence_restart.data", "restart",
    dic={"nx": NX, "ny": NY, "dt": DT, "t_half": T_HALF, "t_final": T_FINAL,
         "sauv": "./pre_run/magnetic_turbulence_pre_pb.sauv"},
    nbProcs=1, excluNR=True, targetData="magnetic_turbulence_restart.data",
    pre_run=pre_run)

ref_case = run.addCaseFromTemplate(
    "magnetic_turbulence_restart_ref.data", "ref",
    dic={"nx": NX, "ny": NY, "dt": DT, "t_final": T_FINAL},
    nbProcs=1, excluNR=True, targetData="magnetic_turbulence_restart_ref.data")

run.printCases()


In [ ]:
run.runCases(verbose=True)
print("Restart and reference cases completed.")


In [ ]:
import pathlib
build_dir = pathlib.Path(BUILD)
fields = ("K", "EPSILON", "B")
sonde_names = {"K": "SONDE_K", "EPSILON": "SONDE_EPSILON", "B": "SONDE_B"}

def last_value(case, sonde):
    probe = build_dir / case.dir_ / f"{case.dataFileName_}_{sonde}.son"
    samples = np.atleast_2d(np.loadtxt(probe, comments="#"))
    return float(samples[-1, 1]), float(samples[-1, 0])

restart_vals = {}
ref_vals = {}
for field in fields:
    rv, rt = last_value(restart_case, sonde_names[field])
    fv, ft = last_value(ref_case, sonde_names[field])
    restart_vals[field] = rv
    ref_vals[field] = fv
    rel = abs(rv - fv) / max(abs(fv), 1e-30)
    print(f"{field:8s}  restart={rv:.8e} (t={rt:.4f})  "
          f"ref={fv:.8e} (t={ft:.4f})  rel.diff={rel:.3e}")
    assert rel < 1e-2, f"{field}: restart does not match reference (rel.diff {rel:.3e})"

print("Restart persistence: PASS (k_b, epsilon_b, magnetic field match to 1e-2).")
print("Note: k_b/epsilon_b restart is not bit-exact (~1e-3 relative); B is exact (uniform).")


In [ ]:
run.tablePerf()
